In [35]:
import os
import sys
import pyarrow as pa
from pathlib import Path
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.feature_selection import mutual_info_regression
from sklearn.model_selection import TimeSeriesSplit
from tqdm import tqdm
from joblib import Parallel, delayed
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
sys.path.insert(0, "..")
from paths import resolve

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
_NCPU = os.cpu_count() or 1
pa.set_cpu_count(_NCPU)
pa.set_io_thread_count(_NCPU)
os.environ["NUMEXPR_MAX_THREADS"] = str(_NCPU)
os.environ["NUMEXPR_NUM_THREADS"] = str(_NCPU)
os.environ.setdefault("OMP_NUM_THREADS", str(_NCPU))
os.environ.setdefault("OPENBLAS_NUM_THREADS", str(_NCPU))
os.environ.setdefault("MKL_NUM_THREADS", str(_NCPU))
print(f"Running with {_NCPU} CPU cores | pyarrow {pa.__version__}", flush=True)

Running with 8 CPU cores | pyarrow 24.0.0


Variables

In [ ]:
%run 0_variables.ipynb

Load raw data

In [ ]:
def load_data():
    def load_features(
            feature_dataset_start: pd.Timestamp,
            feature_dataset_end: pd.Timestamp,
    ) -> pd.DataFrame:
        features = pd.read_parquet(os.environ["FEATURES_PATH"], filters=[
            ('SETTLEMENTDATE', '>=', feature_dataset_start),
            ('SETTLEMENTDATE', '<=', feature_dataset_end),
        ])
        
        features = features.drop(columns=[c for c in features.columns if c in set(os.environ["TARGET_COLS"].split(","))])
        return features.loc[:feature_dataset_end]


    def load_targets(target:str):
        future_prediction_targets = pd.read_parquet(os.environ["TARGETS_PATH"] )
        return future_prediction_targets


    # Make calls
    features = load_features(
        feature_dataset_start=pd.Timestamp(os.environ["FEATURE_DATASET_START"]),
        feature_dataset_end=pd.Timestamp(os.environ["FEATIRE_DATASET_END"]),
    )

    future_prediction_targets = load_targets(
        target=os.environ["TARGET"]
    )

    # Align on index
    future_prediction_targets = future_prediction_targets.loc[features.index]
    features = features.loc[future_prediction_targets.index]

    
    display(features[:3])
    display(future_prediction_targets[:3])

    return features, future_prediction_targets

features, future_prediction_targets = load_data()
features.to_parquet(os.environ["FEATURES_PROCESSED_PATH"])
print(f"Saved features → {os.environ['FEATURES_PROCESSED_PATH']}", flush=True)